### Required Discussion 19:1: Building a Recommender System with SURPRISE

This discussion focuses on exploring additional algorithms with the `Suprise` library to generate recommendations.  Your goal is to identify the optimal algorithm by minimizing the mean squared error using cross validation. You are also going to select a dataset to use from [grouplens](https://grouplens.org/datasets/movielens/) example datasets.  

To begin, head over to [grouplens](https://grouplens.org/datasets/movielens/) and examine the different datasets available.  Choose one so that it is easy to create the data as expected in `Surprise` with user, item, and rating information.  Then, compare the performance of at least the `KNNBasic`, `SVD`, `NMF`, `SlopeOne`, and `CoClustering` algorithms to build your recommendations.  For more information on the algorithms see the documentation for the algorithm package [here](https://surprise.readthedocs.io/en/stable/prediction_algorithms_package.html).

Share the results of your investigation and include the results of your cross validation and a basic description of your dataset with your peers.


In [41]:
import numpy as np
import pandas as pd
import plotly.express as px
from surprise import Dataset, Reader, SVD, NMF, KNNBasic, SlopeOne, CoClustering
from surprise.model_selection import cross_validate, GridSearchCV

In [13]:
df = pd.read_csv('data/ratings.csv')
df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [14]:
 # (0.5 stars - 5.0 stars)
reader = Reader(rating_scale=(0.5, 5))
sf = Dataset.load_from_df(df.drop('timestamp', axis=1), reader=reader)

In [15]:
scores = {}

In [18]:
# SVD
params = {
    'n_factors': range(2, 10, 2),
    'n_epochs': [100, 200],
    'lr_all': [0.005, 0.01, 0.0005],
    'random_state': [42]
}
grid_svd = GridSearchCV(SVD, param_grid=params, cv=3, measures=['mse'])
grid_svd.fit(sf)

In [31]:
scores['SVD'] = {'best_score': grid_svd.best_score['mse'], 'best_params': grid_svd.best_params['mse'],
                 'best_estimator': grid_svd.best_estimator['mse']}

In [32]:
scores

{'SVD': {'best_score': 0.7628712160243665,
  'best_params': {'n_factors': 2, 'n_epochs': 200, 'lr_all': 0.0005},
  'best_estimator': <surprise.prediction_algorithms.matrix_factorization.SVD at 0x765d5ea85be0>},
 'NMF': {'best_score': 0.9075181683658716,
  'best_params': {'n_factors': 2,
   'n_epochs': 200,
   'reg_pu': 0.01,
   'random_state': 42}},
 'KNN': {'best_score': 0.9016288058124416,
  'best_params': {'k': 8, 'min_k': 3}},
 'CoClustering': {'best_score': 0.9021709234757916,
  'best_params': {'n_cltr_u': 2,
   'n_cltr_i': 4,
   'n_epochs': 200,
   'random_state': 42}}}

In [24]:
# NMF
params = {
    'n_factors': range(2, 10, 2),
    'n_epochs': [100, 200],
    'reg_pu': [0.005, 0.01, 0.0005],
    'random_state': [42]
}
grid_nmf = GridSearchCV(NMF, param_grid=params, cv=3, measures=['mse'])
grid_nmf.fit(sf)

In [33]:
scores['NMF'] = {'best_score': grid_nmf.best_score['mse'], 'best_params': grid_nmf.best_params['mse'],
                 'best_estimator': grid_nmf.best_estimator['mse']}

In [26]:
# KNN
params = {
    'k': range(2, 10, 2),
    'min_k': [3, 5],
}
grid_knn = GridSearchCV(KNNBasic, param_grid=params, cv=3, measures=['mse'])
grid_knn.fit(sf)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

In [34]:
scores['KNN'] = {'best_score': grid_knn.best_score['mse'], 'best_params': grid_knn.best_params['mse'],
                 'best_estimator': grid_knn.best_estimator['mse']}

In [29]:
# CoClustering
params = {
    'n_cltr_u': range(2, 6, 2),
    'n_cltr_i': range(2, 6, 2),
    'n_epochs': [100, 200],
    'random_state': [42]
}
grid_cc = GridSearchCV(CoClustering, param_grid=params, cv=3, measures=['mse'])
grid_cc.fit(sf)

In [35]:
scores['CoClustering'] = {'best_score': grid_cc.best_score['mse'], 'best_params': grid_cc.best_params['mse'],
                          'best_estimator': grid_cc.best_estimator['mse']}

In [38]:
scores['SlopeOne'] = {'best_estimator': SlopeOne()}

In [39]:
cross_vals = {}
for key, val in scores.items():
    result = cross_validate(val['best_estimator'], sf, measures=['mse'], cv=5)
    cross_vals[key] = result['test_mse']
cross_vals

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.


{'SVD': array([0.75181472, 0.75900282, 0.7568843 , 0.74958548, 0.75605572]),
 'NMF': array([0.88678032, 0.88977127, 0.88358078, 0.87076736, 0.85922137]),
 'KNN': array([0.87062888, 0.87712672, 0.86943364, 0.89114486, 0.88607506]),
 'CoClustering': array([0.89263934, 0.89422099, 0.87252624, 0.89605252, 0.89695888]),
 'SlopeOne': array([0.81802718, 0.81106436, 0.80572253, 0.81989203, 0.80361052])}

In [42]:
for key, val in cross_vals.items():
    cross_vals[key] = np.mean(val)

cross_vals

{'SVD': 0.754668607299773,
 'NMF': 0.8780242186247383,
 'KNN': 0.8788818323964757,
 'CoClustering': 0.8904795942913506,
 'SlopeOne': 0.8116633231127477}

In [45]:
cv_df = pd.DataFrame.from_dict(cross_vals, orient='index')
cv_df

,0
SVD,0.754669
NMF,0.878024
KNN,0.878882
CoClustering,0.890480
SlopeOne,0.811663


In [52]:
fig = px.bar(cv_df, title='SVD had the best performance', labels={'index': 'Model', 'value': 'Avg Test MSE'})
fig.write_image('images/score.png')
fig.show()

In [51]:
scores

{'SVD': {'best_score': 0.7628712160243665,
  'best_params': {'n_factors': 2, 'n_epochs': 200, 'lr_all': 0.0005},
  'best_estimator': <surprise.prediction_algorithms.matrix_factorization.SVD at 0x765d5ea85be0>},
 'NMF': {'best_score': 0.9075181683658716,
  'best_params': {'n_factors': 2,
   'n_epochs': 200,
   'reg_pu': 0.01,
   'random_state': 42},
  'best_estimator': <surprise.prediction_algorithms.matrix_factorization.NMF at 0x765d60417800>},
 'KNN': {'best_score': 0.9016288058124416,
  'best_params': {'k': 8, 'min_k': 3},
  'best_estimator': <surprise.prediction_algorithms.knns.KNNBasic at 0x765d5d3e7470>},
 'CoClustering': {'best_score': 0.9021709234757916,
  'best_params': {'n_cltr_u': 2,
   'n_cltr_i': 4,
   'n_epochs': 200,
   'random_state': 42},
  'best_estimator': <surprise.prediction_algorithms.co_clustering.CoClustering at 0x765d5da099d0>},
 'SlopeOne': {'best_estimator': <surprise.prediction_algorithms.slope_one.SlopeOne at 0x765d5d3f5970>}}